# Week 08 - Day 01 | Decorators

**Topics:** First-class functions · Basic decorators · `@functools.wraps` · Stacked decorators · Decorator factories

---

## Decorator Cheatsheet

| Pattern | Syntax | Notes |
|---|---|---|
| Basic decorator | `@my_dec` | Sugar for `func = my_dec(func)` |
| Preserve metadata | `@functools.wraps(func)` | Always add inside every decorator |
| Stack decorators | `@dec1` then `@dec2` | Applied bottom-up: `dec1(dec2(f))` |
| Decorator factory | `@repeat(3)` | Sugar for `func = repeat(3)(func)` |
| Wrapper signature | `*args, **kwargs` | Forwards any arguments to original |

In [ ]:
# 1. Functions as first-class objects

def greet(name):
    return f"Hello, {name}!"

say_hello = greet               # assign to variable
print(say_hello("Alice"))       # Hello, Alice!

def apply_twice(func, value):   # pass as argument
    return func(func(value))

print(apply_twice(str.upper, "python"))  # PYTHON

def make_adder(n):              # return a function
    def adder(x):
        return x + n
    return adder

add5 = make_adder(5)
print(add5(10), add5(20))       # 15 25

In [ ]:
# 2. Basic decorator — manual vs @ syntax

def shout(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

# Manual application:
def greet2(name):
    return f"Hello, {name}!"

greet2 = shout(greet2)
print(greet2("world"))    # HELLO, WORLD!

# @ syntax (identical result, cleaner code):
@shout
def greet3(name):
    return f"Hi, {name}!"

print(greet3("Bob"))      # HI, BOB!

In [ ]:
# 3. functools.wraps — preserving metadata
import functools

def naive_dec(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@naive_dec
def compute():
    """Performs a computation."""
    pass

print("Without @wraps:")
print(f"  __name__ = {compute.__name__}")  # wrapper  <- WRONG
print(f"  __doc__  = {compute.__doc__}")   # None     <- WRONG

def proper_dec(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@proper_dec
def compute2():
    """Performs a computation."""
    pass

print("With @functools.wraps:")
print(f"  __name__ = {compute2.__name__}")  # compute2
print(f"  __doc__  = {compute2.__doc__}")   # Performs a computation.

In [ ]:
# 4. Stacked decorators
import functools

def bold(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f"**{func(*args, **kwargs)}**"
    return wrapper

def italic(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f"_{func(*args, **kwargs)}_"
    return wrapper

@bold
@italic
def title(text):
    return text

print(title("Python"))   # **_Python_**

@italic
@bold
def title2(text):
    return text

print(title2("Python"))  # _**Python**_

In [ ]:
# 5. Decorator factory — decorator with arguments
import functools

def repeat(n):                          # factory receives the argument
    def decorator(func):                # real decorator
        @functools.wraps(func)
        def wrapper(*args, **kwargs):   # wrapper
            result = None
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def knock():
    print("Knock!")

knock()
# Knock!
# Knock!
# Knock!

In [ ]:
# 6. Practical decorators: timer + validator
import functools
import time

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed * 1000:.2f}ms")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

print(slow_sum(2_000_000))

def require_positive(func):
    @functools.wraps(func)
    def wrapper(n, *args, **kwargs):
        if not isinstance(n, (int, float)) or n <= 0:
            raise ValueError(f"{func.__name__}: expected positive number, got {n!r}")
        return func(n, *args, **kwargs)
    return wrapper

@require_positive
def sqrt(n):
    return n ** 0.5

print(sqrt(25))
try:
    sqrt(-4)
except ValueError as e:
    print(e)

## Summary

| Concept | Key Point |
|---|---|
| Decorator | Function that takes a function and returns a new function |
| `@my_dec` | Sugar for `func = my_dec(func)` |
| `*args, **kwargs` | Always use in wrapper to forward any call signature |
| `@functools.wraps` | Always add — preserves `__name__`, `__doc__`, etc. |
| Stacking | Applied bottom-up; `@A @B def f` means `A(B(f))` |
| Factory | Extra nesting level to pass arguments: `repeat(3)` returns the decorator |

> **Next:** Day 02 — Iterators (`__iter__`, `__next__`, custom iterators)